# 02 - Processing Geometries

Prepares the spatial boundary files used throughout the project. Loads SF neighborhood boundaries from SF OpenData and simplifies them to just name and geometry. Clips Census tract boundaries and EPC tract data to the SF city boundary. Exports cleaned GeoJSONs for neighborhoods, census tracts, EPC tracts, block groups, and the SF city polygon.

In [1]:
import geopandas as gpd

In [2]:
# neighborhood boundaries from SF OpenData
sf_neigh = gpd.read_file('../../data/raw/sf_neighborhoods.geojson')

In [3]:
# only keeping neighborhood name
sf_neigh_filt = sf_neigh[['nhood', 'geometry']].rename(columns={'nhood': 'neighborhood'})

In [4]:
# export to processed polygons folder
sf_neigh_filt.to_file('../../data/processed/polygons/sf_neighborhoods.geojson', driver='GeoJSON')

In [5]:
# Importing SF geometry
# URL for 2025 TIGER/Line Place boundaries
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]

sf_poly = sf_poly.to_crs(epsg=4326)

In [6]:
# Census cartographic boundary file for CA tracts (2020)
tracts_gdf = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_06_tract_500k.zip"
)

In [7]:
# simplifying
tracts_gdf = tracts_gdf[["NAME", "NAMELSAD", "STATE_NAME", "GEOID", "geometry"]]
tracts_gdf = tracts_gdf.drop(columns=["index_right"], errors="ignore")

In [8]:
sf_tracts = gpd.clip(tracts_gdf.to_crs(sf_poly.crs), sf_poly)

In [9]:
epc_tracts = gpd.read_file("../../data/raw/epc_tracts.geojson")
epc_tracts_sf = gpd.clip(epc_tracts.to_crs(sf_poly.crs), sf_poly)
epc_tracts_sf.to_file('../../data/processed/polygons/epc_tracts_sf.geojson', driver='GeoJSON')
sf_poly.to_file('../../data/processed/polygons/sf_poly.geojson', driver='GeoJSON')

In [10]:
sf_tracts.to_file('../../data/processed/polygons/sf_tracts.geojson', driver='GeoJSON')

In [11]:
sf_block_groups = gpd.read_file('../../data/raw/sf_block_groups.geojson')
sf_block_groups = sf_block_groups.rename(columns={'geoid': 'GEOID'})

In [12]:
cols = [
    'blkgrpce', 'GEOID', 'namelsad', 'geometry'
]
sf_block_groups[cols].to_file('../../data/processed/polygons/sf_block_grp.geojson', driver='GeoJSON')